In [1]:
# ==========================================
# K-Means Clustering with NLP
# Dataset: people-1000.csv
# ==========================================

# Install packages (run once if needed)
# !pip install pandas numpy scikit-learn nltk matplotlib

import pandas as pd
import numpy as np
import nltk
import string

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import matplotlib.pyplot as plt

# Download NLTK resources (run once)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# ------------------------------------------
# Load Dataset
# ------------------------------------------
df = pd.read_csv("people-1000.csv")

print(df.head())
print(df.columns)

# ------------------------------------------
# Select text column
# Replace "job" with your column name
# ------------------------------------------
text_column = "job"

df[text_column] = df[text_column].fillna("").astype(str)

# ------------------------------------------
# NLP Preprocessing
# ------------------------------------------
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))

    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

df["processed_text"] = df[text_column].apply(preprocess)

# ------------------------------------------
# TF-IDF Vectorization
# ------------------------------------------
vectorizer = TfidfVectorizer(max_features=1000)

X = vectorizer.fit_transform(df["processed_text"])

print("TF-IDF Shape:", X.shape)

# ------------------------------------------
# Find Best K (Elbow Method)
# ------------------------------------------
inertia = []

K = range(2, 11)

for k in K:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X)
    inertia.append(model.inertia_)

plt.figure(figsize=(8,5))
plt.plot(K, inertia, marker='o')
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.grid(True)
plt.show()

# ------------------------------------------
# Train K-Means
# ------------------------------------------
k = 5

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X)

df["Cluster"] = clusters

# ------------------------------------------
# Evaluate
# ------------------------------------------
score = silhouette_score(X, clusters)

print("Silhouette Score:", score)

# ------------------------------------------
# Top Keywords per Cluster
# ------------------------------------------
terms = vectorizer.get_feature_names_out()

order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

for i in range(k):
    print(f"\nCluster {i}")

    keywords = [terms[ind] for ind in order_centroids[i, :10]]

    print(", ".join(keywords))

# ------------------------------------------
# Show Sample Records
# ------------------------------------------
for i in range(k):
    print("\n================================")
    print(f"Cluster {i}")
    print(df[df["Cluster"] == i][[text_column]].head())

# ------------------------------------------
# Save Results
# ------------------------------------------
df.to_csv("people-1000-clustered.csv", index=False)

print("\nClustered dataset saved as people-1000-clustered.csv")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


   Index          User Id First Name Last Name     Sex  \
0      1  8717bbf45cCDbEe     Shelia   Mahoney    Male   
1      2  3d5AD30A4cD38ed         Jo    Rivers  Female   
2      3  810Ce0F276Badec     Sheryl    Lowery  Female   
3      4  BF2a889C00f0cE1    Whitney    Hooper    Male   
4      5  9afFEafAe1CBBB9    Lindsey      Rice  Female   

                           Email               Phone Date of birth  \
0            pwarner@example.org        857.139.8239    2014-01-27   
1  fergusonkatherine@example.net     +1-950-759-8687    1931-07-26   
2            fhoward@example.org       (599)782-0605    2013-11-25   
3          zjohnston@example.com     +1-939-130-6258    2012-11-17   
4               elin@example.net  (390)417-1635x3010    1923-04-15   

                  Job Title  
0         Probation officer  
1                    Dancer  
2                      Copy  
3  Counselling psychologist  
4       Biomedical engineer  
Index(['Index', 'User Id', 'First Name', 'Last Nam

KeyError: 'job'